In [13]:
import polars as pl
import polars_h3 as plh3
from pathlib import Path
import yaml

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

path = f"{config['dirs']['output']}/{config['city']}"

In [ ]:
result = {7: pl.DataFrame(), 8: pl.DataFrame(), 9: pl.DataFrame(), 10: pl.DataFrame()}
for i in range(256):
    data = pl.read_parquet(Path(f"{path}/{i:02x}/partitioned/pings.parquet"))

    data = data.with_columns(
        plh3.int_to_str(
            plh3.latlng_to_cell(pl.col("latitude"), pl.col("longitude"), 7)
        ).alias("h3_7"),
        plh3.int_to_str(
            plh3.latlng_to_cell(pl.col("latitude"), pl.col("longitude"), 8)
        ).alias("h3_8"),
        plh3.int_to_str(
            plh3.latlng_to_cell(pl.col("latitude"), pl.col("longitude"), 9)
        ).alias("h3_9"),
        plh3.int_to_str(
            plh3.latlng_to_cell(pl.col("latitude"), pl.col("longitude"), 10)
        ).alias("h3_10"),
    )

    for res in range(7, 11):
        result[res] = pl.concat(
            [
                result[res],
                data.group_by([f"h3_{res}"])
                .len()
                .rename({f"h3_{res}": "h3", "len": "count"}),
            ]
        )
# runs 24m for Budapest region

In [14]:
for res in range(7, 11):
    result[res].group_by(["h3"]).agg(pl.col.count.sum()).sort("h3").write_csv(
        f"{path}/descriptives/activity_count_h3_{res}.csv"
    )